# Basic SQL ETL and Schema Walkthrough

This notebook is a notebook version of a small Airflow DAG. It loads source data into an offline database, pulls the result set back with SQL, transforms it, trains real SuperGLM model revisions, writes model outputs back to the database, and then changes the deployed package.

The offline database is SQLite so this runs anywhere. The production pattern is not SQLite-specific: the same DAG shape can target SQL Server, Azure SQL with Entra auth, Postgres, DuckDB, or another database. You only change the connection helper and the DDL dialect when the target database requires different syntax.

## Migration Resources And ERD Files

`pricing_pipeline.resources.migrations` is the authoritative SQL Server schema. The next cell lists its installed migration resources rather than relying on a copied DDL extract.

For ERDs, use the maintained Mermaid diagrams in `docs/sql/diagrams` or run `uv run python scripts/render_schema_diagrams.py`. It writes ignored output on demand; do not commit generated runnable SQL.

`pricing.PREDICT_CURRENT_RATE` is supplied by the packaged migration chain; it is a stored procedure rather than a table, so most ERD tools will not draw it.

## Provider Note

SQLite is only the offline stand-in for this tutorial. At work you might point the exact same task sequence at Azure SQL with Entra auth, Azure SQL with Entra token auth, username/password SQL Server, Postgres, DuckDB, or another database. The stable parts are the task boundaries and the audit model: load data, record the dataset, train the model, persist the model run, persist the rate package, and choose a deployment. The parts that change are the connection helper and the DDL dialect.

## What Counts As A Model Revision

A model revision is not just a changed Python file. Treat these as revisions because they can change predictions or audit meaning:

- Changed feature list
- Changed training SQL or dataset window
- Changed preprocessing
- Changed model class or hyperparameters
- Changed target, offset, or sample-weight definition
- Changed manual rating package
- Changed deployment slot pointer

The database keeps historical packages. A deployment change only changes what is current for a slot; historical packages remain available for audit.

## Notebook DAG: Offline Load, Train, Push, Deploy

The cells below intentionally mirror an Airflow DAG:

1. Create or reset the offline database.
2. Push source rows into `raw.FREMTPL_RAW`.
3. Pull a training result set with SQL.
4. Transform the frame for model training.
5. Train model revision 1 and push outputs.
6. Train model revision 2 with a changed feature spec and push outputs.
7. Pick which package to deploy.
8. Create a Manual 10% Uplift Package.
9. Deploy the manual package instead.
10. Query current and historical state back from SQL.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)
os.chdir(REPO_ROOT)
REPO_ROOT

In [ ]:
import json
import re
import shutil
import sqlite3
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import text
from superglm import Categorical, Numeric, SuperGLM

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

from pricing_pipeline.infra.config import Settings
from pricing_pipeline.infra.db import get_engine
from pricing_pipeline.resources import migration_root

## Inspect The Installed Migration Resources

The executable SQLite demo below uses a compatible subset of the installed SQL Server migration contract. Mermaid diagrams and the rendering command described above support ERD inspection without copied runnable SQL.

In [ ]:
migration_names = tuple(
    item.name
    for item in sorted(migration_root().iterdir(), key=lambda item: item.name)
    if item.is_file() and item.name.startswith("V")
)
display(migration_names)

## Offline Database Setup

SQLite supports attached databases, so this tutorial can use logical schema names such as `raw.FREMTPL_RAW`, `mlops.MODEL_RUN`, and `pricing.RATE_PACKAGE`. That keeps the SQL close to the real SQL Server shape while still running offline.

In [ ]:
OFFLINE_DB_ROOT = REPO_ROOT / "state" / "tutorials" / "offline_pricing_lab"
SCHEMA_NAMES = ("raw", "mlops", "pricing", "pricing_runtime")


def reset_offline_db_root() -> None:
    if OFFLINE_DB_ROOT.exists():
        shutil.rmtree(OFFLINE_DB_ROOT)
    OFFLINE_DB_ROOT.mkdir(parents=True, exist_ok=True)


def attach_pricing_lab_schemas(con: sqlite3.Connection) -> None:
    for schema_name in SCHEMA_NAMES:
        db_path = OFFLINE_DB_ROOT / f"{schema_name}.sqlite"
        escaped_path = str(db_path).replace("'", "''")
        con.execute(f"ATTACH DATABASE '{escaped_path}' AS {schema_name}")


def connect_offline_db() -> sqlite3.Connection:
    con = sqlite3.connect(OFFLINE_DB_ROOT / "main.sqlite")
    con.execute("PRAGMA foreign_keys = ON")
    attach_pricing_lab_schemas(con)
    return con


def create_offline_schema(con: sqlite3.Connection) -> None:
    con.executescript(
        """
        CREATE TABLE raw.FREMTPL_RAW (
            IDpol INTEGER PRIMARY KEY,
            AccountingMonth TEXT NOT NULL,
            ClaimNb INTEGER NOT NULL,
            Exposure REAL NOT NULL,
            Area TEXT,
            VehPower INTEGER,
            VehAge INTEGER,
            DrivAge INTEGER,
            BonusMalus INTEGER,
            VehBrand TEXT,
            VehGas TEXT,
            Density REAL,
            Region TEXT
        );

        CREATE TABLE mlops.DATASET_MANIFEST (
            manifest_id TEXT PRIMARY KEY,
            dataset_name TEXT NOT NULL,
            source_schema TEXT,
            source_table TEXT,
            source_system TEXT,
            data_as_of_date TEXT NOT NULL,
            row_count INTEGER NOT NULL,
            pk_columns_json TEXT NOT NULL,
            target_column TEXT,
            weight_column TEXT,
            created_ts TEXT NOT NULL,
            created_by TEXT NOT NULL
        );

        CREATE TABLE mlops.DATASET_COLUMN (
            manifest_id TEXT NOT NULL,
            ordinal_no INTEGER NOT NULL,
            column_name TEXT NOT NULL,
            column_role TEXT NOT NULL,
            pandas_dtype TEXT NOT NULL,
            null_count INTEGER NOT NULL,
            distinct_count INTEGER,
            PRIMARY KEY (manifest_id, ordinal_no)
        );

        CREATE TABLE pricing.MODEL (
            model_id INTEGER PRIMARY KEY AUTOINCREMENT,
            model_name TEXT UNIQUE NOT NULL,
            model_label TEXT,
            target_name TEXT NOT NULL,
            model_type TEXT NOT NULL,
            model_status TEXT NOT NULL,
            created_ts TEXT NOT NULL,
            created_by TEXT NOT NULL,
            retired_ts TEXT
        );

        CREATE TABLE mlops.MODEL_RUN (
            model_run_id INTEGER PRIMARY KEY AUTOINCREMENT,
            model_id INTEGER NOT NULL,
            dag_id TEXT NOT NULL,
            airflow_run_id TEXT NOT NULL,
            mlflow_experiment_id TEXT,
            mlflow_run_id TEXT,
            model_version TEXT,
            run_status TEXT NOT NULL,
            started_ts TEXT NOT NULL,
            completed_ts TEXT,
            created_by TEXT NOT NULL,
            feature_spec_json TEXT NOT NULL
        );

        CREATE TABLE mlops.MODEL_RUN_METRIC (
            model_run_id INTEGER NOT NULL,
            metric_name TEXT NOT NULL,
            metric_value REAL NOT NULL,
            metric_scope TEXT,
            PRIMARY KEY (model_run_id, metric_name)
        );

        CREATE TABLE pricing.RATE_PACKAGE (
            rate_package_id INTEGER PRIMARY KEY AUTOINCREMENT,
            parent_rate_package_id INTEGER,
            model_id INTEGER NOT NULL,
            model_run_id INTEGER,
            model_version TEXT,
            package_version INTEGER NOT NULL,
            base_rate REAL NOT NULL,
            effective_from_date TEXT NOT NULL,
            effective_to_date TEXT,
            package_status TEXT NOT NULL,
            revision_reason TEXT NOT NULL,
            created_ts TEXT NOT NULL,
            created_by TEXT NOT NULL
        );

        CREATE TABLE pricing.MODEL_DEPLOYMENT (
            deployment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            model_id INTEGER NOT NULL,
            rate_package_id INTEGER NOT NULL,
            deployment_slot TEXT NOT NULL,
            effective_from_ts TEXT NOT NULL,
            effective_to_ts TEXT,
            deployed_by TEXT NOT NULL,
            deployment_note TEXT,
            created_ts TEXT NOT NULL
        );

        CREATE TABLE pricing_runtime.MODEL_SCORE (
            model_run_id INTEGER,
            rate_package_id INTEGER NOT NULL,
            policy_id INTEGER NOT NULL,
            model_name TEXT NOT NULL,
            model_version TEXT NOT NULL,
            score_type TEXT NOT NULL,
            exposure REAL NOT NULL,
            veh_age INTEGER NOT NULL,
            driver_age INTEGER NOT NULL,
            rating_area TEXT NOT NULL,
            prediction REAL NOT NULL,
            adjustment_factor REAL NOT NULL,
            PRIMARY KEY (rate_package_id, policy_id, score_type)
        );
        """
    )


reset_offline_db_root()
con = connect_offline_db()
create_offline_schema(con)
print(f"offline_db_root={OFFLINE_DB_ROOT}")

## Task 1: Load Source Data From Somewhere

In real Airflow this could be a `SELECT` from another SQL Server database, a file landing task, or an API extract. Here we generate a deterministic freMTPL-shaped source dataset and push it into `raw.FREMTPL_RAW`.

In [ ]:
def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_frame(con: sqlite3.Connection, schema_name: str, table_name: str, frame: pd.DataFrame) -> None:
    columns = list(frame.columns)
    quoted_columns = ", ".join(f'"{column}"' for column in columns)
    placeholders = ", ".join("?" for _ in columns)
    sql = f"INSERT INTO {schema_name}.{table_name} ({quoted_columns}) VALUES ({placeholders})"
    rows = [tuple(sqlite_value(row[column]) for column in columns) for _, row in frame.iterrows()]
    con.executemany(sql, rows)
    con.commit()


def make_source_rows(row_count: int = 180, seed: int = 20260512) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    area = rng.choice(["A", "B", "C"], size=row_count, p=[0.45, 0.35, 0.20])
    veh_age = rng.integers(0, 18, size=row_count)
    driv_age = rng.integers(18, 82, size=row_count)
    exposure = rng.uniform(0.2, 1.0, size=row_count).round(4)
    density = rng.lognormal(mean=5.8, sigma=0.7, size=row_count).round(1)
    area_effect = pd.Series(area).map({"A": -0.20, "B": 0.05, "C": 0.35}).to_numpy()
    eta = -2.25 + 0.045 * veh_age + 0.008 * (driv_age - 45) + area_effect + np.log(exposure)
    claim_nb = rng.poisson(np.exp(eta))
    month = np.where(np.arange(row_count) < 120, "2025-01", "2025-02")

    return pd.DataFrame(
        {
            "IDpol": np.arange(1, row_count + 1),
            "AccountingMonth": month,
            "ClaimNb": claim_nb,
            "Exposure": exposure,
            "Area": area,
            "VehPower": rng.integers(4, 12, size=row_count),
            "VehAge": veh_age,
            "DrivAge": driv_age,
            "BonusMalus": rng.integers(50, 160, size=row_count),
            "VehBrand": rng.choice(["B1", "B2", "B3"], size=row_count),
            "VehGas": rng.choice(["Regular", "Diesel"], size=row_count),
            "Density": density,
            "Region": rng.choice(["R1", "R2", "R3", "R4"], size=row_count),
        }
    )


def seed_source_rows(con: sqlite3.Connection) -> pd.DataFrame:
    source_rows = make_source_rows()
    append_frame(con, "raw", "FREMTPL_RAW", source_rows)
    return source_rows


loaded_source_rows = seed_source_rows(con)
pd.read_sql_query("SELECT COUNT(*) AS source_rows_loaded FROM raw.FREMTPL_RAW", con)

## Task 2: Pull The Training Result Set With SQL

This is the thing you would usually write as the first model-specific task. Notice it is a normal SQL result set, not a hardcoded dataframe.

In [ ]:
TRAINING_SQL = """
SELECT
    IDpol,
    AccountingMonth,
    ClaimNb,
    Exposure,
    Area,
    VehAge,
    DrivAge,
    Density
FROM raw.FREMTPL_RAW
WHERE Exposure > 0
  AND AccountingMonth BETWEEN :start_month AND :end_month
ORDER BY IDpol
"""


def read_training_result_set(
    con: sqlite3.Connection,
    *,
    start_month: str,
    end_month: str,
) -> pd.DataFrame:
    return pd.read_sql_query(
        TRAINING_SQL,
        con,
        params={"start_month": start_month, "end_month": end_month},
    )


source_result_set = read_training_result_set(con, start_month="2025-01", end_month="2025-02")
source_result_set.head(10)

## Task 3: Transform The Result Set For Modelling

This mirrors the model-specific preprocessing code. The second revision below will change the feature spec, which is exactly the kind of change that should create a new model run and package.

In [ ]:
def transform_source_rows(raw: pd.DataFrame) -> pd.DataFrame:
    transformed = raw.copy()
    transformed["LogDensity"] = np.log(transformed["Density"].astype(float).clip(lower=1.0))
    transformed["VehAgeBand"] = pd.cut(
        transformed["VehAge"],
        bins=[0, 1, 3, 7, 15, 100],
        right=False,
    ).astype(str)
    return transformed


transformed_result_set = transform_source_rows(source_result_set)
transformed_result_set.head(10)

## Task 4: Register Dataset Audit Rows

This is the minimal dataset audit part of the DAG. It records what source table was used and what columns were present before any model run consumes it.

In [ ]:
def record_dataset_manifest(
    con: sqlite3.Connection,
    frame: pd.DataFrame,
    *,
    manifest_id: str,
    dataset_name: str,
) -> str:
    now = datetime.now(UTC).isoformat(timespec="seconds")
    manifest = pd.DataFrame(
        [
            {
                "manifest_id": manifest_id,
                "dataset_name": dataset_name,
                "source_schema": "raw",
                "source_table": "FREMTPL_RAW",
                "source_system": "offline_sqlite_demo",
                "data_as_of_date": now[:10],
                "row_count": len(frame),
                "pk_columns_json": json.dumps(["IDpol"]),
                "target_column": "ClaimNb",
                "weight_column": "Exposure",
                "created_ts": now,
                "created_by": "notebook",
            }
        ]
    )
    columns = pd.DataFrame(
        {
            "manifest_id": manifest_id,
            "ordinal_no": np.arange(1, len(frame.columns) + 1),
            "column_name": frame.columns,
            "column_role": [
                "KEY" if column == "IDpol" else "TARGET" if column == "ClaimNb" else "WEIGHT" if column == "Exposure" else "FEATURE"
                for column in frame.columns
            ],
            "pandas_dtype": frame.dtypes.astype(str).to_numpy(),
            "null_count": frame.isna().sum().astype(int).to_numpy(),
            "distinct_count": frame.nunique(dropna=True).astype(int).to_numpy(),
        }
    )
    append_frame(con, "mlops", "DATASET_MANIFEST", manifest)
    append_frame(con, "mlops", "DATASET_COLUMN", columns)
    return manifest_id


manifest_id = record_dataset_manifest(
    con,
    source_result_set,
    manifest_id="freMTPL_demo_20260512",
    dataset_name="freMTPL_offline_demo",
)
pd.read_sql_query("SELECT * FROM mlops.DATASET_MANIFEST", con)

## Task 5: Train Real SuperGLM Revisions

Revision 1 uses `VehAge` and `LogDensity`. Revision 2 changes the feature spec by adding `DrivAge` and `Area`. That is a model spec change, so it gets a separate `mlops.MODEL_RUN` and a separate `pricing.RATE_PACKAGE`.

In [ ]:
FEATURE_LIBRARY = {
    "VehAge": Numeric(),
    "DrivAge": Numeric(),
    "LogDensity": Numeric(),
    "Area": Categorical(),
}


def build_superglm_model(feature_columns: list[str]) -> SuperGLM:
    return SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        discrete=True,
        n_bins=32,
        features={feature: FEATURE_LIBRARY[feature] for feature in feature_columns},
    )


def training_arrays(frame: pd.DataFrame, feature_columns: list[str]):
    X = frame.loc[:, feature_columns].copy()
    y = frame["ClaimNb"].to_numpy(dtype=float)
    exposure = frame["Exposure"].to_numpy(dtype=float)
    offset = np.log(exposure)
    return X, y, exposure, offset


def train_superglm_revision(
    frame: pd.DataFrame,
    *,
    model_version: str,
    feature_columns: list[str],
) -> dict:
    X, y, exposure, offset = training_arrays(frame, feature_columns)
    model = build_superglm_model(feature_columns)
    fitted_model = model.fit_reml(X, y, offset=offset)
    predictions = fitted_model.predict(X, offset=offset)
    deviance = float(getattr(getattr(fitted_model, "result", None), "deviance", np.nan))
    return {
        "model_version": model_version,
        "feature_columns": feature_columns,
        "fitted_model": fitted_model,
        "X": X,
        "y": y,
        "exposure": exposure,
        "offset": offset,
        "predictions": predictions,
        "deviance": deviance,
        "row_count": len(frame),
    }


revision_1 = train_superglm_revision(
    transformed_result_set,
    model_version="v1_base_vehage_density",
    feature_columns=["VehAge", "LogDensity"],
)
revision_2 = train_superglm_revision(
    transformed_result_set,
    model_version="v2_add_driver_age_area",
    feature_columns=["VehAge", "DrivAge", "LogDensity", "Area"],
)
pd.DataFrame(
    [
        {
            "model_version": revision_1["model_version"],
            "features": ", ".join(revision_1["feature_columns"]),
            "deviance": revision_1["deviance"],
            "row_count": revision_1["row_count"],
        },
        {
            "model_version": revision_2["model_version"],
            "features": ", ".join(revision_2["feature_columns"]),
            "deviance": revision_2["deviance"],
            "row_count": revision_2["row_count"],
        },
    ]
)

## Task 6: Push Model Runs, Packages, Metrics, And Scores Back To SQL

This is the `load back to SQL` part. The fitted object itself would normally go to MLflow/artifact storage. The database gets audit rows, metrics, package metadata, and queryable outputs.

In [ ]:
def ensure_model(con: sqlite3.Connection) -> int:
    now = datetime.now(UTC).isoformat(timespec="seconds")
    con.execute(
        """
        INSERT OR IGNORE INTO pricing.MODEL
            (model_name, model_label, target_name, model_type, model_status, created_ts, created_by)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        ("MTPL_FREQ", "Motor frequency", "ClaimNb", "superglm_poisson", "ACTIVE", now, "notebook"),
    )
    con.commit()
    return int(
        con.execute("SELECT model_id FROM pricing.MODEL WHERE model_name = ?", ("MTPL_FREQ",)).fetchone()[0]
    )


def next_package_version(con: sqlite3.Connection, model_id: int) -> int:
    value = con.execute(
        "SELECT COALESCE(MAX(package_version), 0) + 1 FROM pricing.RATE_PACKAGE WHERE model_id = ?",
        (model_id,),
    ).fetchone()[0]
    return int(value)


def persist_model_revision(
    con: sqlite3.Connection,
    revision: dict,
    frame: pd.DataFrame,
    *,
    model_id: int,
    manifest_id: str,
    revision_reason: str,
) -> dict:
    now = datetime.now(UTC).isoformat(timespec="seconds")
    feature_spec_json = json.dumps(
        {
            "manifest_id": manifest_id,
            "feature_columns": revision["feature_columns"],
            "offset": "log(Exposure)",
            "target": "ClaimNb",
        },
        sort_keys=True,
    )
    con.execute(
        """
        INSERT INTO mlops.MODEL_RUN
            (model_id, dag_id, airflow_run_id, mlflow_experiment_id, mlflow_run_id,
             model_version, run_status, started_ts, completed_ts, created_by, feature_spec_json)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            model_id,
            "notebook_mtpl_frequency_demo",
            f"offline__{revision['model_version']}",
            None,
            None,
            revision["model_version"],
            "SUCCEEDED",
            now,
            now,
            "notebook",
            feature_spec_json,
        ),
    )
    model_run_id = int(con.execute("SELECT last_insert_rowid()").fetchone()[0])

    metric_rows = pd.DataFrame(
        [
            {"model_run_id": model_run_id, "metric_name": "deviance", "metric_value": revision["deviance"], "metric_scope": "train"},
            {"model_run_id": model_run_id, "metric_name": "row_count", "metric_value": revision["row_count"], "metric_scope": "train"},
        ]
    )
    append_frame(con, "mlops", "MODEL_RUN_METRIC", metric_rows)

    package_version = next_package_version(con, model_id)
    base_rate = float(frame["ClaimNb"].sum() / frame["Exposure"].sum())
    con.execute(
        """
        INSERT INTO pricing.RATE_PACKAGE
            (parent_rate_package_id, model_id, model_run_id, model_version, package_version,
             base_rate, effective_from_date, effective_to_date, package_status, revision_reason,
             created_ts, created_by)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            None,
            model_id,
            model_run_id,
            revision["model_version"],
            package_version,
            base_rate,
            now[:10],
            None,
            "PUBLISHED",
            revision_reason,
            now,
            "notebook",
        ),
    )
    rate_package_id = int(con.execute("SELECT last_insert_rowid()").fetchone()[0])

    score_rows = frame.loc[:, ["IDpol", "Exposure", "VehAge", "DrivAge", "Area"]].copy()
    score_rows = score_rows.rename(
        columns={"IDpol": "policy_id", "Exposure": "exposure", "VehAge": "veh_age", "DrivAge": "driver_age", "Area": "rating_area"}
    )
    score_rows["model_run_id"] = model_run_id
    score_rows["rate_package_id"] = rate_package_id
    score_rows["model_name"] = "MTPL_FREQ"
    score_rows["model_version"] = revision["model_version"]
    score_rows["score_type"] = "MODEL_SCORE"
    score_rows["prediction"] = revision["predictions"]
    score_rows["adjustment_factor"] = 1.0
    score_rows = score_rows.loc[
        :,
        [
            "model_run_id",
            "rate_package_id",
            "policy_id",
            "model_name",
            "model_version",
            "score_type",
            "exposure",
            "veh_age",
            "driver_age",
            "rating_area",
            "prediction",
            "adjustment_factor",
        ],
    ]
    append_frame(con, "pricing_runtime", "MODEL_SCORE", score_rows)

    con.commit()
    return {
        "model_run_id": model_run_id,
        "rate_package_id": rate_package_id,
        "package_version": package_version,
    }


model_id = ensure_model(con)
revision_1_ids = persist_model_revision(
    con,
    revision_1,
    transformed_result_set,
    model_id=model_id,
    manifest_id=manifest_id,
    revision_reason="Initial base feature spec",
)
revision_2_ids = persist_model_revision(
    con,
    revision_2,
    transformed_result_set,
    model_id=model_id,
    manifest_id=manifest_id,
    revision_reason="Changed feature list: added DrivAge and Area",
)
pd.read_sql_query(
    """
    SELECT
        p.rate_package_id,
        p.package_version,
        p.model_version,
        p.revision_reason,
        m.metric_value AS train_deviance
    FROM pricing.RATE_PACKAGE AS p
    JOIN mlops.MODEL_RUN_METRIC AS m
        ON m.model_run_id = p.model_run_id
       AND m.metric_name = 'deviance'
    ORDER BY p.package_version
    """,
    con,
)

## Task 7: Choose Which Model Package To Deploy

This is the deployment decision point. Nothing is deleted. We look at candidate packages and choose which package should be current for a deployment slot.

In [ ]:
def deploy_rate_package(
    con: sqlite3.Connection,
    *,
    model_id: int,
    rate_package_id: int,
    deployment_slot: str,
    deployed_by: str,
    deployment_note: str,
) -> None:
    now = datetime.now(UTC).isoformat(timespec="seconds")
    con.execute(
        """
        UPDATE pricing.MODEL_DEPLOYMENT
        SET effective_to_ts = ?
        WHERE model_id = ?
          AND deployment_slot = ?
          AND effective_to_ts IS NULL
        """,
        (now, model_id, deployment_slot),
    )
    con.execute(
        """
        INSERT INTO pricing.MODEL_DEPLOYMENT
            (model_id, rate_package_id, deployment_slot, effective_from_ts, effective_to_ts,
             deployed_by, deployment_note, created_ts)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (model_id, rate_package_id, deployment_slot, now, None, deployed_by, deployment_note, now),
    )
    con.commit()


def current_deployment(con: sqlite3.Connection, deployment_slot: str = "UAT") -> pd.DataFrame:
    return pd.read_sql_query(
        """
        SELECT
            d.deployment_id,
            d.deployment_slot,
            m.model_name,
            p.rate_package_id,
            p.package_version,
            p.model_version,
            p.revision_reason,
            p.parent_rate_package_id,
            d.effective_from_ts,
            d.deployed_by,
            d.deployment_note
        FROM pricing.MODEL_DEPLOYMENT AS d
        JOIN pricing.RATE_PACKAGE AS p
            ON p.rate_package_id = d.rate_package_id
           AND p.model_id = d.model_id
        JOIN pricing.MODEL AS m
            ON m.model_id = d.model_id
        WHERE d.deployment_slot = :deployment_slot
          AND d.effective_to_ts IS NULL
        """,
        con,
        params={"deployment_slot": deployment_slot},
    )


def deployment_history(con: sqlite3.Connection) -> pd.DataFrame:
    return pd.read_sql_query(
        """
        SELECT
            d.deployment_id,
            d.deployment_slot,
            p.rate_package_id,
            p.package_version,
            p.model_version,
            p.revision_reason,
            d.effective_from_ts,
            d.effective_to_ts,
            d.deployment_note
        FROM pricing.MODEL_DEPLOYMENT AS d
        JOIN pricing.RATE_PACKAGE AS p
            ON p.rate_package_id = d.rate_package_id
        ORDER BY d.deployment_id
        """,
        con,
    )


candidate_packages = pd.read_sql_query(
    """
    SELECT
        p.rate_package_id,
        p.package_version,
        p.model_version,
        p.revision_reason,
        m.metric_value AS train_deviance
    FROM pricing.RATE_PACKAGE AS p
    JOIN mlops.MODEL_RUN_METRIC AS m
        ON m.model_run_id = p.model_run_id
       AND m.metric_name = 'deviance'
    ORDER BY m.metric_value ASC
    """,
    con,
)
display(candidate_packages)

chosen_package_id = int(revision_2_ids["rate_package_id"])
deploy_rate_package(
    con,
    model_id=model_id,
    rate_package_id=chosen_package_id,
    deployment_slot="UAT",
    deployed_by="notebook",
    deployment_note="Deploy v2 because the spec includes driver age and area signal",
)
current_deployment(con, "UAT")

## Task 8: Manual 10% Uplift Package

Now pretend a business review decides that old vehicle ages need a 10% uplift. We do not mutate the deployed package. We create a child package from the deployed package, copy the score rows, apply the uplift, and deploy that child package. This demonstrates how the deployed package changes while historical packages remain.

In [ ]:
def create_manual_uplift_package(
    con: sqlite3.Connection,
    *,
    parent_rate_package_id: int,
    uplift_factor: float = 1.10,
    min_vehicle_age: int = 10,
) -> int:
    now = datetime.now(UTC).isoformat(timespec="seconds")
    parent = con.execute(
        """
        SELECT model_id, model_run_id, model_version, base_rate
        FROM pricing.RATE_PACKAGE
        WHERE rate_package_id = ?
        """,
        (parent_rate_package_id,),
    ).fetchone()
    if parent is None:
        raise ValueError(f"Unknown parent package {parent_rate_package_id}")

    parent_model_id, parent_model_run_id, parent_model_version, parent_base_rate = parent
    package_version = next_package_version(con, int(parent_model_id))
    manual_version = f"{parent_model_version}_manual_vehage10pct"
    con.execute(
        """
        INSERT INTO pricing.RATE_PACKAGE
            (parent_rate_package_id, model_id, model_run_id, model_version, package_version,
             base_rate, effective_from_date, effective_to_date, package_status, revision_reason,
             created_ts, created_by)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            parent_rate_package_id,
            parent_model_id,
            parent_model_run_id,
            manual_version,
            package_version,
            parent_base_rate,
            now[:10],
            None,
            "PUBLISHED",
            f"Manual 10% uplift where VehAge >= {min_vehicle_age}",
            now,
            "notebook",
        ),
    )
    manual_rate_package_id = int(con.execute("SELECT last_insert_rowid()").fetchone()[0])

    parent_scores = pd.read_sql_query(
        """
        SELECT
            model_run_id,
            policy_id,
            model_name,
            exposure,
            veh_age,
            driver_age,
            rating_area,
            prediction
        FROM pricing_runtime.MODEL_SCORE
        WHERE rate_package_id = :parent_rate_package_id
          AND score_type = 'MODEL_SCORE'
        ORDER BY policy_id
        """,
        con,
        params={"parent_rate_package_id": parent_rate_package_id},
    )
    parent_scores["rate_package_id"] = manual_rate_package_id
    parent_scores["model_version"] = manual_version
    parent_scores["score_type"] = "MODEL_SCORE"
    parent_scores["adjustment_factor"] = np.where(parent_scores["veh_age"] >= min_vehicle_age, uplift_factor, 1.0)
    parent_scores["prediction"] = parent_scores["prediction"] * parent_scores["adjustment_factor"]
    parent_scores = parent_scores.loc[
        :,
        [
            "model_run_id",
            "rate_package_id",
            "policy_id",
            "model_name",
            "model_version",
            "score_type",
            "exposure",
            "veh_age",
            "driver_age",
            "rating_area",
            "prediction",
            "adjustment_factor",
        ],
    ]
    append_frame(con, "pricing_runtime", "MODEL_SCORE", parent_scores)
    return manual_rate_package_id


manual_package_id = create_manual_uplift_package(
    con,
    parent_rate_package_id=chosen_package_id,
    uplift_factor=1.10,
    min_vehicle_age=10,
)
manual_comparison = pd.read_sql_query(
    """
    SELECT
        base.policy_id,
        base.veh_age,
        base.prediction AS v2_prediction,
        manual.prediction AS manual_prediction,
        manual.adjustment_factor
    FROM pricing_runtime.MODEL_SCORE AS base
    JOIN pricing_runtime.MODEL_SCORE AS manual
        ON manual.policy_id = base.policy_id
    WHERE base.rate_package_id = :base_package_id
      AND manual.rate_package_id = :manual_package_id
      AND base.veh_age >= 10
    ORDER BY base.policy_id
    LIMIT 10
    """,
    con,
    params={"base_package_id": chosen_package_id, "manual_package_id": manual_package_id},
)
manual_comparison

## Task 9: Deploy The Manual Package Instead

The deployment slot now points at the manual package. The previous deployment row is expired with `effective_to_ts`; it is not deleted.

In [ ]:
deploy_rate_package(
    con,
    model_id=model_id,
    rate_package_id=manual_package_id,
    deployment_slot="UAT",
    deployed_by="notebook",
    deployment_note="Use manual 10% VehAge uplift package instead of raw v2 model package",
)

display(current_deployment(con, "UAT"))
display(deployment_history(con))

## Task 10: Pull The Current Deployed Scores Back From SQL

This is what a downstream consumer would do: query the package currently deployed to a slot, then pull scores or rating outputs for that package.

In [ ]:
current_deployed_scores = pd.read_sql_query(
    """
    SELECT
        s.policy_id,
        s.model_name,
        s.model_version,
        s.rate_package_id,
        s.veh_age,
        s.rating_area,
        s.exposure,
        s.prediction,
        s.adjustment_factor
    FROM pricing_runtime.MODEL_SCORE AS s
    JOIN pricing.MODEL_DEPLOYMENT AS d
        ON d.rate_package_id = s.rate_package_id
    WHERE d.deployment_slot = 'UAT'
      AND d.effective_to_ts IS NULL
    ORDER BY s.policy_id
    LIMIT 15
    """,
    con,
)
current_deployed_scores

## Real SQL Server / Azure SQL Version

The offline notebook used SQLite so the whole DAG is executable without Docker or a corporate database. In the real project, the task boundaries stay the same and the engine changes.

For SQL Server or Azure SQL, the load call usually becomes `DataFrame.to_sql(..., schema=target_schema, if_exists='append')` through SQLAlchemy. For Azure SQL with Entra auth, hide token handling in `pricing_pipeline/infra/db.py` so model-specific code still only asks for an engine.

For Postgres or DuckDB, use the same model-specific ETL functions but adjust DDL types, identity syntax, filtered indexes, and auth/connection details to that database's DDL dialect.

In [ ]:
def run_real_sql_server_task(
    *,
    from_month: str,
    source_database: str,
    target_database: str,
    target_schema: str,
    target_table: str,
) -> int:
    settings = Settings.from_env(os.environ)
    source_engine = get_engine(settings, database=source_database)
    target_engine = get_engine(settings, database=target_database)

    raw = pd.read_sql_query(
        text(TRAINING_SQL),
        source_engine,
        params={"start_month": from_month, "end_month": "9999-12"},
    )
    transformed = transform_source_rows(raw)

    with target_engine.begin() as sql_server_connection:
        transformed.to_sql(
            target_table,
            sql_server_connection,
            schema=target_schema,
            if_exists="append",
            index=False,
            chunksize=10_000,
        )

    return len(transformed)

## What This Proves

- We loaded data from a SQL-like source into `raw.FREMTPL_RAW`.
- We pulled a SQL result set into pandas.
- We transformed that result set.
- We trained real SuperGLM model revisions.
- We pushed model runs, metrics, packages, and scores back to SQL tables.
- We chose one package to deploy.
- We created a Manual 10% Uplift Package and deployed that instead.
- The deployed package changes, while historical packages remain available for audit.